# Modelo realista · sin datos del futuro (Pablo y Adri · `realista/`)

> **Qué es "realista":** el modelo solo ve el **snapshot actual y snapshots PASADOS** de la sesión, como funcionaría un
> asistente en tiempo real. Sin snapshots posteriores y sin el oráculo (etiquetas de otros controladores). No es `main`
> (la regla 1 de `main` solo permite el snapshot) ni `futuro/`.

**Este notebook junta** el diario de Pablo (antes `LGB_realista.ipynb`: bloques de features e intentos de mejora) y el de
Adri (antes `LGB_realista_adri.ipynb`: búsqueda de mejoras sobre `real_todo`), más la **búsqueda nocturna** de Adri
(`realista/noche_adri.py`). Las celdas de código **no entrenan nada**: leen los OOF guardados y calculan las métricas.
Los entrenamientos se lanzan desde la terminal (comandos en cada sección).

**Código** (reutiliza por importación funciones de `futuro/` sin modificarlas):

| Archivo | Autor | Qué hace |
|---|---|---|
| `features_real.py` | Pablo | columnas que solo miran atrás; un `assert` falla si se cuela una que mira al futuro (`blk_len`, `gap_next`, `gxc_exitfl_next`) |
| `feat_causal.py` | Pablo | bloque `causal` (`cz_*`): segundos desde el último cambio del estado que resuelve cada tarea |
| `feat_hist.py` | Pablo | bloque `hist` (`h_*`): tareas resueltas / nuevas respecto al snapshot anterior (no aporta) |
| `feat_session_causal.py` | Pablo | segundo paso de sesión que solo usa scores de snapshots anteriores (sin fuga: comprobado) |
| `m_lgb_real.py` | Pablo | LightGBM realista; guarda en `realista/oof|preds|submissions` y apunta en `experiments/lgb_real_runs.csv` |
| `m_gbm_real.py`, `stack_real.py`, `tune_real.py` | Pablo | XGBoost/CatBoost, stacking con `main` y búsqueda Optuna |
| **`noche_adri.py`** | Adri | búsqueda larga y reanudable: parámetros, confirmación con 3 semillas, mezclas y versión final |

**Bloques de features:** `base` = columnas originales (las `a_*`, `x_*`, `k_*` del reto ya miran solo atrás) ·
`logic` = reglas dentro del snapshot · `past` = `p_*`, `blk_pos`, `gap_prev` · `causal` = `cz_*` · `geo` = velocidad de
acercamiento al borde, tráfico que cruza el mismo borde y niveles (de `futuro/feat_geo.py`, sin lo que mira al futuro).

**Cómo leer las cifras:** `cv` = top-3 out-of-fold con `Data/folds.csv`. Con 1 semilla, diferencias < ~0.005 son ruido.
Se adopta un cambio si sube la CV **y** suben más sesiones de las que bajan (regla del equipo, sección 14 de futuro).
*CV 023* = acierto en los 6 controladores de la CV más parecidos al 023 del privado (002, 011, 010, 006, 017, 015; Juanma).

> ## Estado actual (21/10, 11:00)
>
> **Mejor modelo realista: `real_adri_noche`** = mezcla a pesos iguales de 5 LightGBM con parámetros de la búsqueda
> nocturna (bloques `base + logic + past + causal + geo`, 4 semillas cada uno, sin segundo paso):
> **CV 0.6732**, CV 023 0.6734, peor controlador 0.595. Frente a `real_todo` con los parámetros de siempre (0.6697 en la
> misma prueba): **+0.35**, 12 controladores suben y 6 bajan; por sesiones, empate (17 / 18).
>
> **Para Kaggle: `realista/submissions/sub_real_adri_noche.csv`** (sin `_ses`: aquí el segundo paso resta, 0.6732 → 0.6722).
>
> | Paso | CV | peor controlador | dónde |
> |---|---|---|---|
> | solo el snapshot (`real_base`, 3 semillas) | 0.6572 | 0.574 | 1 |
> | + `logic` + `past` + `causal` + `geo` (`real_todo`, 3 semillas) | 0.6703 (0.6718 con segundo paso) | 0.596 | 1 |
> | parámetros de la búsqueda nocturna, mezcla de 5 (`real_adri_noche`) | **0.6732** | 0.595 | 4 |
>
> **Lo que sabemos (no repetir):**
> 1. **Lo que aporta es el pasado del avión:** cuánto hace que cambió el estado ligado a la tarea (`causal`, +0.7) y su
>    movimiento reciente y el tráfico en el borde (`geo`, +0.6). `logic` y `past` no aportan (1).
> 2. **Sin futuro, el segundo paso de sesión apenas ayuda** (−0.3 a +0.3): el "explaining away" necesita saber qué pasa después (1, 4).
> 3. **No suman:** stack con los modelos de `main`, `hist`, XGBoost, CatBoost (2); peso por tramo, quitar `a_n300`/`a_n60`/`seq_len`,
>    `instr` y atención hacia atrás, modo del controlador, 63 hojas + `extra_trees` (3, 4). La mitad hacia atrás del bloque
>    `instr` de Juanma empeora: lo que le funciona con futuro es la *siguiente* instrucción.
> 4. **Lo único que sube son los parámetros** (Optuna de Pablo → búsqueda nocturna): lr ~0.02 con 550–1000 árboles, ~37 hojas,
>    `max_bin=63`, `reg_lambda` ~21, subsample ~0.94, **sin** `extra_trees`. Con 1 semilla parecía +0.5–0.6; confirmado con
>    3 semillas, **+0.2–0.4** (4).
> 5. **El ruido de semilla es del mismo tamaño que las mejoras:** los parámetros de siempre dan 0.6623–0.6631 con 1 semilla y
>    0.6697–0.6703 con 3. Nada se adopta sin confirmarlo con 3 o más semillas.
> 6. **El techo realista está en ~0.67.** Con futuro, el mismo LightGBM llega a ~0.91: el salto lo da saber qué hace el
>    controlador después (siguiente instrucción, cuándo desaparece la tarea).

## Preparación
Solo lee archivos (~10 s): OOF de `realista/oof/`, registros de `experiments/lgb_real_runs.csv` y `realista/noche/`.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "realista" else Path.cwd()
REAL = ROOT / "realista"
sys.path[:0] = [str(ROOT / "futuro"), str(ROOT / "cv")]
import metricas
from metric import hits

INFO = metricas.info()                       # sample_id -> atco, fold, session, scenario
CV023 = [2, 11, 10, 6, 17, 15]               # controladores parecidos al 023 (Juanma)


def oof(nombre):
    p = REAL / "oof" / f"{nombre}.csv"
    return pd.read_csv(p) if p.exists() else None


def resumen(nombres, ref=None):
    """Métricas robustas de cada OOF de realista/oof (y frente a ref, si se da)."""
    r, filas = (oof(ref) if ref else None), []
    for n in nombres:
        o = oof(n)
        if o is None:
            filas.append(dict(modelo=n, nota="no está en realista/oof/"))
            continue
        m = metricas.metricas(o, r if n != ref else None)
        h = hits(o)
        filas.append(dict(modelo=n, cv=m["cv"], cv_023=h[INFO.atco.reindex(h.index).isin(CV023).values].mean(),
                          peor_controlador=m["min_atco"], mediana_sesion=m["ses_median"],
                          esc1=m["sc1"], esc2=m["sc2"], esc3=m["sc3"], d_cv=m.get("d_cv"),
                          sesiones_suben_bajan=f"{m['ses_up']}/{m['ses_down']}" if "ses_up" in m else "",
                          controladores_suben_bajan=f"{m['atco_up']}/{m['atco_down']}" if "atco_up" in m else ""))
    return pd.DataFrame(filas).round(4)


RUNS = pd.read_csv(ROOT / "experiments" / "lgb_real_runs.csv")
print(f"{len(RUNS)} ejecuciones en experiments/lgb_real_runs.csv · OOF disponibles:",
      sorted(p.stem for p in (REAL / "oof").glob("*.csv")))

24 ejecuciones en experiments/lgb_real_runs.csv · OOF disponibles: ['real_adri', 'real_adri_noche', 'real_adri_noche_ses', 'real_adri_ses']


## 1 · Bloques de features (Pablo, 21/10 02:20)

**Hipótesis:** sin futuro, el pasado del avión (cómo ha cambiado su estado, cómo se mueve) debería ayudar a saber qué
tarea toca. Se añaden los bloques de uno en uno, 3 semillas, con y sin el segundo paso causal.

| Modelo | Bloques | 1.er pase | + segundo paso causal | peor controlador (+ sesión) | frente a `real_base_ses`: sesiones / controladores suben–bajan |
|---|---|---|---|---|---|
| `real_base` | base | 0.6572 | 0.6556 | 0.568 (016) | — |
| `real_blp` | + logic + past | 0.6571 | 0.6597 | 0.591 (016) | 20/17 · 10/8 |
| `real_blpc` | + causal | 0.6645 | 0.6617 | 0.590 (016) | 26/11 · 11/6 |
| **`real_todo`** | **+ geo** | **0.6703** | **0.6718** | **0.600 (016)** | **28/9 · 15/3** |

**Conclusión bloque 1 · el pasado del avión aporta +1.5 sobre el snapshot solo.**
- `causal` (+0.7) y `geo` (+0.6) son lo que aporta; `logic` y `past`, nada.
- El segundo paso causal apenas cambia (−0.3 a +0.3): sin futuro pierde casi toda su fuerza (en futuro daba +0.7).
- Con futuro, el mismo LightGBM llega a ~0.91: el resto del salto es saber qué hace el controlador después.

Reproducir (terminal, desde la raíz; ~8 min cada uno):
```
python realista/m_lgb_real.py --sesion --name real_base --feats base
python realista/m_lgb_real.py --sesion --name real_todo --feats base,logic,past,causal,geo --ref real_base
```

In [2]:
# Registro de Pablo (los OOF de real_base/real_todo se generaron en gpu.dsic y no están en el repo)
RUNS[RUNS.name.str.startswith("real_") & ~RUNS.name.str.startswith("real_adri")][
    ["when", "name", "cv", "min_atco", "ses_median", "sc1", "sc2", "sc3", "d_cv", "ses_up", "ses_down", "seeds"]].round(4)

,when,name,cv,min_atco,ses_median,sc1,sc2,sc3,d_cv,ses_up,ses_down,seeds
0,2026-10-21 02:23,real_base,0.6572,0.5741,0.6608,0.6614,0.6496,0.6612,NaN,NaN,NaN,3
1,2026-10-21 02:24,real_base_ses,0.6556,0.5685,0.6573,0.6612,0.6466,0.6594,NaN,NaN,NaN,3
2,2026-10-21 02:24,real_blp,0.6571,0.5910,0.6765,0.6599,0.6433,0.6695,-0.0002,20.0,16.0,3
3,2026-10-21 02:25,real_blp_ses,0.6597,0.5910,0.6753,0.6618,0.6457,0.6733,0.0042,20.0,17.0,3
4,2026-10-21 02:25,real_blpc,0.6645,0.5947,0.6694,0.6724,0.6507,0.6713,0.0073,23.0,13.0,3
5,2026-10-21 02:26,real_blpc_ses,0.6617,0.5901,0.6612,0.6671,0.6503,0.6686,0.0061,26.0,11.0,3
6,2026-10-21 02:26,real_todo,0.6703,0.5957,0.6785,0.6689,0.6675,0.6750,0.0131,25.0,11.0,3
7,2026-10-21 02:26,real_todo_ses,0.6718,0.6004,0.6838,0.6728,0.6667,0.6765,0.0162,28.0,9.0,3
10,2026-10-21 02:55,real_hist,0.6641,0.5938,0.6739,0.6662,0.6538,0.6733,-0.0062,13.0,22.0,3
11,2026-10-21 02:56,real_hist_ses,0.6641,0.5891,0.6679,0.6667,0.6533,0.6735,-0.0077,14.0,21.0,3


## 2 · Intentos de mejora de Pablo (21/10, 02:35–03:05)

Referencia: `real_todo` = 0.6703 · `real_todo_ses` = 0.6718 (3 semillas). Ninguna feature nueva mira al futuro: **prueba de
fuga sobre TODAS las features realistas** (truncar cada sesión al 40 % y al 70 % y recalcular) → 0 filas distintas.

| Prueba | Archivo | CV | + 2.º paso | frente a `real_todo(_ses)`: sesiones · controladores | Veredicto |
|---|---|---|---|---|---|
| Stack logit con los modelos de `main` (nn_tf_v2, lgb_adri, linear, types) | `stack_real.py` | 0.6670 | 0.6678 | 16/18 · 6/11 | ✗ |
| Mezclas simples `real_todo` + 0.1–0.5 × (nn / lgb_adri / linear) | — | 0.668–0.6726 | — | ±0.1 | ✗ ruido |
| `hist` (`h_*`): tareas resueltas / nuevas respecto al snapshot anterior | `feat_hist.py` | 0.6641 | 0.6641 | 13/22 · 8/10 | ✗ |
| XGBoost rank:ndcg (GPU) | `m_gbm_real.py` | 0.6651 | 0.6654 | 12/20 · 5/12 | ✗ |
| `real_todo` + 0.25 × XGBoost | — | 0.6721 | — | 20/13 | ≈ ruido |
| CatBoost YetiRank (GPU) | `m_gbm_real.py` | 0.6398 | 0.6429 | 6/31 · 0/18 | ✗ |
| **Optuna sobre el LightGBM** (28 pruebas, 1 semilla) | `tune_real.py` → `realista/tune/t1.log` | **0.6684** (1 s) | — | — | **prometedor** → se sigue en 4 |

**Conclusión bloque 2 · solo los parámetros apuntan a mejora.** Las 5 mejores pruebas de Optuna coinciden: `lambdarank`,
lr baja (~0.015–0.02) con ~1100–1400 árboles, `max_bin=63`, subsample ~0.9, colsample ~0.75–0.9, 30–50 hojas,
truncación 5–8. Con 1 semilla, +0.6: había que confirmarlo con más semillas (se hace en 4).

Reproducir:
```
python realista/tune_real.py --study t1 --trials 60 --threads 8
python realista/m_gbm_real.py --algo xgb --name xgb_todo --sesion --ref real_todo
python realista/stack_real.py real_todo nn_tf_v2 lgb_adri linear types --name stk_a --sesion --nosave
```

## 3 · Búsqueda rápida de mejoras sobre `real_todo` (Adri, 21/10 03:15)

**Hipótesis:** lo que funcionó en el modelo de un snapshot (`notebooks/LGB_adri.ipynb`) y en futuro debería ayudar aquí.
Búsqueda automática (1 semilla), cada candidata sobre lo mejor hasta ese momento; se queda si sube la CV ≥ 0.003 **y**
suben más sesiones de las que bajan.

| Candidata | De dónde sale | CV (1 semilla) | Δ vs `real_todo` (0.6631) | sesiones suben / bajan | CV 023 | |
|---|---|---|---|---|---|---|
| `real_todo` (referencia) | Pablo | 0.6631 | — | — | 0.6556 | |
| peso por tramo (1/√longitud al entrenar) | `LGB_adri` 12c: +0.8 con un snapshot | 0.6636 | +0.0005 | 19 / 15 | 0.6536 | ruido |
| quitar `a_n300`, `a_n60`, `seq_len` | Juanma `h4` | 0.6606 | −0.0024 | 15 / 19 | 0.6554 | ✗ |
| `instr` hacia atrás + atención reciente | Juanma, futuro 21 (mitad causal) | 0.6559 | −0.0071 | 13 / 23 | 0.6490 | ✗ |
| lo anterior + 63 hojas + `extra_trees` | Juanma, futuro 21 | 0.6557 | −0.0074 | 14 / 21 | 0.6479 | ✗ |
| modo del controlador (tipo de su última instrucción) | análisis de fallos de `LGB_adri` | 0.6575 | −0.0056 | 12 / 23 | 0.6504 | ✗ |
| 63 hojas + `extra_trees` | Juanma, futuro 13a | 0.6579 | −0.0052 | 17 / 18 | 0.6508 | ✗ |
| parámetros de Optuna (mejor prueba de Pablo) | sección 2 | 0.6660 | +0.0029 | 15 / 21 | 0.6589 | casi |

Versión resultante (`real_adri`: `real_todo` tal cual, 5 semillas): CV 0.6686, con segundo paso 0.6690 (Pablo: 0.6703 /
0.6718 con 3 semillas; la diferencia es ruido de semillas).

**Conclusión bloque 3 · nada de lo que funcionó en otros modelos se traslada al realista.**
- El peso por tramo no aporta aquí: las features de pasado ya le dicen al modelo lo que el peso corregía.
- La mitad hacia atrás de `instr` empeora: con futuro lo que funciona es la *siguiente* instrucción, no la anterior.
- Solo los parámetros de Optuna suben algo → se exploran a fondo en la sección 4.
- El código de esta búsqueda (antes en `LGB_realista_adri.ipynb`) queda en el historial de git; lo sustituye `noche_adri.py`.

## 4 · Búsqueda nocturna de parámetros (Adri, `noche_adri.py`, 21/10 03:45–06:39)

**Hipótesis:** los parámetros son lo único que mueve la nota (2, 3). Hay que explorarlos más y **confirmar con varias
semillas** lo que con 1 semilla parece ganar.

**Etapas** (`realista/noche_adri.py`, 2.9 h; informe completo en `realista/noche/INFORME.md`, registro en
`realista/noche/resultados.jsonl`):
1. referencia: `real_todo` con 3 semillas → **0.6697** (Pablo: 0.6703);
2. búsqueda: 63 pruebas propias (mutaciones de las mejores) + 19 del Optuna de Pablo, 1 semilla;
3. confirmación de las 6 mejores con 3 semillas;
4. mezclas (media de z-scores en el snapshot, pesos iguales) de las 2–5 mejores;
5. extras sobre la mejor: `hist`, `instr` + atención, peso por tramo;
6. versión final: la mezcla de 5, con 4 semillas por miembro, y el segundo paso causal.

**4a · Confirmación con 3 semillas** (frente a `real_todo` con 3 semillas, 0.6697)

| Configuración | CV con 1 semilla | **CV con 3 semillas** | Δ | CV 023 | peor controlador | sesiones suben / bajan | controladores |
|---|---|---|---|---|---|---|---|
| `b045` | 0.6721 | **0.6732** | +0.0036 | 0.6719 | **0.601** | 18 / 15 | 12 / 6 |
| `b030` | 0.6748 | 0.6731 | +0.0034 | 0.6675 | 0.594 | 19 / 13 | 10 / 7 |
| `b056` | 0.6733 | 0.6721 | +0.0024 | 0.6690 | 0.597 | 18 / 15 | 12 / 5 |
| `b059` | 0.6728 | 0.6717 | +0.0020 | 0.6699 | 0.593 | 19 / 17 | 9 / 9 |
| `b060` | 0.6717 | 0.6714 | +0.0017 | 0.6673 | 0.594 | 17 / 17 | 7 / 8 |
| `b013` (con `extra_trees`) | 0.6718 | 0.6683 | −0.0014 | 0.6602 | 0.591 | 16 / 19 | 9 / 8 |

Las cinco que confirman se parecen: lr 0.019–0.026, 540–1010 árboles, 36–37 hojas, `min_child` 16–25, `max_bin=63` (127 en `b060`),
`reg_lambda` ~21.6, subsample 0.94, colsample 0.77, truncación 5–8 y **sin `extra_trees`** (la única con `extra_trees`, `b013`,
no confirma). Ejemplo, `b045`: `n_estimators=756, lr=0.0186, leaves=37, min_child=25, subsample=0.94, colsample=0.77,
reg_lambda=21.6, min_gain=0.0005, max_bin=63, cat_smooth=1.5, extra_trees=False, trunc=8`.

**4b · Mezclas** (media de z-scores, pesos iguales)

| Mezcla | CV | Δ | CV 023 | peor controlador | sesiones suben / bajan |
|---|---|---|---|---|---|
| top-2 | 0.6737 | +0.0040 | 0.6727 | 0.596 | 17 / 16 |
| top-3 | 0.6740 | +0.0043 | 0.6736 | 0.598 | 18 / 17 |
| top-4 | 0.6734 | +0.0037 | 0.6719 | 0.597 | 19 / 15 |
| **top-5** | **0.6742** | **+0.0046** | 0.6730 | 0.597 | 18 / 15 |

**4c · Extras sobre `b045`** (2 semillas, frente a `b045` con las mismas): `hist` −0.0059, `instr` + atención −0.0055, peso
por tramo −0.0024. **Ninguno se queda.**

**4d · Versión final `real_adri_noche`** (mezcla de los 5, 4 semillas por miembro; frente a `real_todo` 3 semillas)

| | CV | CV 023 | peor controlador | mediana por sesión | Δ | sesiones suben / bajan | controladores suben / bajan |
|---|---|---|---|---|---|---|---|
| **`real_adri_noche`** | **0.6732** | **0.6734** | 0.595 | 0.6743 | **+0.0036** | 17 / 18 | **12 / 6** |
| `real_adri_noche_ses` | 0.6722 | 0.6719 | 0.595 | 0.6763 | +0.0025 | 17 / 17 | 12 / 6 |

**Conclusión bloque 4 · los parámetros dan una mejora pequeña y real: +0.35, sobre todo por controladores.**
- Lo que con 1 semilla parecía +0.5 queda en **+0.2–0.4** con 3: la mitad era suerte de la semilla.
- La mezcla de 5 no supera claramente a la mejor configuración sola (mezcla 0.6742 frente a `b045` solo 0.6732, ambos con
  3 semillas por miembro): aporta estabilidad más que nota.
- Suben 12 de 18 controladores y la CV 023 sube +0.3; por sesiones queda en empate (17 / 18).
- El segundo paso causal **resta** (0.6732 → 0.6722): para entregar, la versión sin `_ses`.

Reproducir (desde la raíz; ~3 h; si se corta, el mismo comando continúa):
```
.venv\Scripts\python.exe realista\noche_adri.py --horas-busqueda 1.5
```

In [3]:
# Versión final frente a real_adri (real_todo con los parámetros de siempre y 5 semillas): solo lee los OOF guardados
resumen(["real_adri", "real_adri_ses", "real_adri_noche", "real_adri_noche_ses"], ref="real_adri")

,modelo,cv,cv_023,peor_controlador,mediana_sesion,esc1,esc2,esc3,d_cv,sesiones_suben_bajan,controladores_suben_bajan
0,real_adri,0.6686,0.6651,0.5938,0.6818,0.6684,0.6630,0.6752,NaN,,
1,real_adri_ses,0.6690,0.6688,0.5976,0.6803,0.6675,0.6654,0.6747,0.0004,16/12,9/8
2,real_adri_noche,0.6732,0.6734,0.5947,0.6743,0.6752,0.6662,0.6789,0.0046,17/17,10/6
3,real_adri_noche_ses,0.6722,0.6719,0.5947,0.6763,0.6739,0.6656,0.6777,0.0036,22/13,13/5


In [4]:
# ¿Dónde mejora? Acierto top-3 según el tipo de la tarea correcta
TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]
tipo = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=["sample_id", "target_index", "task_index"])
                  for f in ("train", "validation")]).query("target_index == 1").set_index("sample_id").task_index
t = pd.DataFrame({n: hits(oof(n)) for n in ("real_adri", "real_adri_noche")})
t["tipo"] = tipo.reindex(t.index).map(dict(enumerate(TIPOS)))
g = t.groupby("tipo").agg(muestras=("real_adri", "size"), real_adri=("real_adri", "mean"),
                          real_adri_noche=("real_adri_noche", "mean"))
g["Δ"] = g.real_adri_noche - g.real_adri
g.sort_values("muestras", ascending=False).round(3)

,muestras,real_adri,real_adri_noche,Δ
tipo,,,,
transfer,3527,0.871,0.872,0.001
assume,3366,0.840,0.835,-0.005
zone,2121,0.717,0.725,0.008
entry_coord,1068,0.413,0.434,0.022
return_to_route,937,0.537,0.555,0.018
exit_coord,745,0.115,0.119,0.004
exit_conditions,448,0.163,0.185,0.022
conflict,390,0.451,0.431,-0.021
qos,257,0.140,0.156,0.016


**Dónde mejora `real_adri_noche`:** entry_coordination +2.2, exit_conditions +2.2, return_to_route +1.8, zone +0.8 y qos +1.6:
sobre todo tareas con instrucción, que según Juanma (futuro 20f) son el margen del 023. Empeora en conflict (−2.1) y
exit_conflict (−2.5), con pocas muestras (390 y 197). non_conformance sube +14 puntos, pero son 86 muestras.

## 5 · Entrega

| Archivo | Qué es | CV |
|---|---|---|
| **`realista/submissions/sub_real_adri_noche.csv`** | **mejor modelo realista** (mezcla de 5, sin segundo paso) | **0.6732** |
| `realista/submissions/sub_real_adri_noche_ses.csv` | lo mismo con el segundo paso causal | 0.6722 |
| `realista/submissions/sub_real_adri.csv` / `_ses.csv` | `real_todo` con los parámetros de siempre, 5 semillas | 0.6686 / 0.6690 |

Las cuatro están comprobadas (1.863 snapshots, una vez cada uno, 3 tareas válidas). Las sube Juanma. Recordatorio: el
público de Kaggle es solo el controlador 020 (±3 puntos de margen); se decide por la CV.

In [5]:
# Comprobación de las submissions
test_ids = set(pd.read_csv(ROOT / "Data" / "test.csv", usecols=["sample_id"]).sample_id)
for p in sorted((REAL / "submissions").glob("sub_real_adri*.csv")):
    s = pd.read_csv(p)
    ok = set(s.sample_id) == test_ids and s.sample_id.is_unique and s.prediction.str.split().str.len().eq(3).all()
    print(f"{p.name:32s} {len(s)} filas · válida: {ok}")

sub_real_adri.csv                1863 filas · válida: True
sub_real_adri_noche.csv          1863 filas · válida: True
sub_real_adri_noche_ses.csv      1863 filas · válida: True
sub_real_adri_ses.csv            1863 filas · válida: True


## 6 · Siguientes ideas (no implementadas)

- **Más semillas en la versión final** (4 → 10 por miembro): +0.1–0.2 esperable. `noche_adri.py --semillas-final 40` reutiliza
  todo lo calculado y solo entrena las semillas nuevas (~30 min; con 5 miembros usa 1.2 × 40 / 5 ≈ 10 por miembro).
- **Reanudar la búsqueda** alrededor de `b045`/`b030` con 2 semillas por prueba (menos ruido que con 1).
- Lo que **no** merece la pena repetir está en *Lo que sabemos* (arriba).

## Historial

In [6]:
# Todas las ejecuciones del modelo realista (Pablo y Adri) y las pruebas de la búsqueda nocturna
cols = ["when", "name", "cv", "min_atco", "ses_median", "sc1", "sc2", "sc3", "d_cv", "ses_up", "ses_down", "seeds", "cv023"]
display(RUNS[[c for c in cols if c in RUNS]].round(4))
noche = pd.DataFrame([json.loads(x) for x in open(REAL / "noche" / "resultados.jsonl", encoding="utf-8")])
noche[["nombre", "etapa", "cv", "cv023", "min_atco", "ses_median", "d_cv", "ses_up", "ses_down"]].round(4)

,when,name,cv,min_atco,ses_median,sc1,sc2,sc3,d_cv,ses_up,ses_down,seeds,cv023
0,2026-10-21 02:23,real_base,0.6572,0.5741,0.6608,0.6614,0.6496,0.6612,NaN,NaN,NaN,3,NaN
1,2026-10-21 02:24,real_base_ses,0.6556,0.5685,0.6573,0.6612,0.6466,0.6594,NaN,NaN,NaN,3,NaN
2,2026-10-21 02:24,real_blp,0.6571,0.5910,0.6765,0.6599,0.6433,0.6695,-0.0002,20.0,16.0,3,NaN
3,2026-10-21 02:25,real_blp_ses,0.6597,0.5910,0.6753,0.6618,0.6457,0.6733,0.0042,20.0,17.0,3,NaN
4,2026-10-21 02:25,real_blpc,0.6645,0.5947,0.6694,0.6724,0.6507,0.6713,0.0073,23.0,13.0,3,NaN
5,2026-10-21 02:26,real_blpc_ses,0.6617,0.5901,0.6612,0.6671,0.6503,0.6686,0.0061,26.0,11.0,3,NaN
6,2026-10-21 02:26,real_todo,0.6703,0.5957,0.6785,0.6689,0.6675,0.6750,0.0131,25.0,11.0,3,NaN
7,2026-10-21 02:26,real_todo_ses,0.6718,0.6004,0.6838,0.6728,0.6667,0.6765,0.0162,28.0,9.0,3,NaN
8,2026-10-21 02:51,xgb_todo,0.6651,0.5872,0.6770,0.6623,0.6566,0.6779,-0.0052,12.0,20.0,3,NaN
9,2026-10-21 02:52,xgb_todo_ses,0.6654,0.5835,0.6783,0.6651,0.6547,0.6777,-0.0065,13.0,23.0,3,NaN


,nombre,etapa,cv,cv023,min_atco,ses_median,d_cv,ses_up,ses_down
0,ref_3s,referencia,0.6697,0.6703,0.5938,0.6799,NaN,NaN,NaN
1,b000,busqueda,0.6646,0.6567,0.5743,0.6645,NaN,NaN,NaN
2,b001,busqueda,0.6696,0.6589,0.6079,0.6775,NaN,NaN,NaN
3,b002,busqueda,0.6682,0.6633,0.5943,0.6739,NaN,NaN,NaN
4,b003,busqueda,0.6643,0.6609,0.5863,0.6748,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
75,b045+hist_2s,extras,0.6676,0.6631,0.5901,0.6710,-0.0059,11.0,19.0
76,b045+instr_atencion_2s,extras,0.6681,0.6635,0.5976,0.6686,-0.0055,9.0,21.0
77,b045+peso_tramo_2s,extras,0.6711,0.6703,0.5857,0.6763,-0.0024,17.0,18.0
78,real_adri_noche,final,0.6732,0.6734,0.5947,0.6743,0.0036,17.0,18.0
